In [ ]:
import numpy
import pandas
import scipy
import uproot

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

# inputs
CALO_FILES = {
  'Run1': None,
  'Run2': '/Users/triozzi/Analysis/icarus-trigger/data/dedx/Run2_dEdx.txt',
  'Run3': None,
  'Run4': None,
  'Run5': None,
}

# grab the KE vs. range PDG table...
file = uproot.open("/Users/triozzi/Analysis/icarus-trigger/data/input/dEdxVsRange_Muon_LArReco.root")
f = scipy.interpolate.interp1d(
  file["dedx_range_mu"].axis().centers(), 
  file["dedx_range_mu"].values(), 
  kind='cubic'
)
xnew = numpy.arange(0.04, 20, 0.01)
ynew = f(xnew)

# dump all plots here, and gitignore it
PRINT_PATH = 'plots/'

In [ ]:
# get calroimetry data from files
VARS = [
  "length", "energy", "end_x", "end_y", "end_z", "mediandedx", "energy_last_20cm", "rr", "dedx"
]

df_W = pandas.read_csv(
  "/Users/triozzi/Analysis/icarus-trigger/data/dedx/Run2_dEdx.txt",
  sep='\t', 
  names=VARS,
)
df_E = pandas.read_csv(
  "/Users/triozzi/Analysis/icarus-trigger/data/dedx/Run2_dEdx_EastCryo.txt",
  sep='\t', 
  names=VARS,
)
df = pandas.concat(
  [df_W, df_E],
  ignore_index=True, 
  sort=False
)

# some filtering...
FILTER_GOOD_DEDX = (df.dedx < 15) & (df.mediandedx > 4) & (df.energy_last_20cm > 60)
rr  = df[FILTER_GOOD_DEDX].rr
dedx = df[FILTER_GOOD_DEDX].dedx

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3), layout='constrained')

im = ax.hist2d(
  rr[(rr > 0) & (rr <= 20)], 
  dedx[(rr > 0) & (rr <= 20)], 
  bins=(50, 30), 
  cmin=0,
  rasterized=True
)

# bethe-bloch reference
ax.plot(xnew, ynew, c='red', lw=2, label='predicted mean d$E$/d$x$')

# labels
ax.set(
  xlabel = 'residual range [cm]',
  ylabel = 'd$E$/d$x$ [MeV/cm]',
  title = 'ICARUS Run2 calibration data'
)

leg = ax.legend(frameon=False, fontsize=12)
for text in leg.get_texts():
    text.set_color("white")

# colorbar
cb_ax = fig.add_axes([0.525, 0.59, 0.3, 0.03])
cbar = fig.colorbar(im[3], cax=cb_ax, orientation='horizontal')
cbar.set_label("Counts [#]", fontsize=12, rotation=0, labelpad=-40, c='white', loc='center')
cb_ax.tick_params(labelsize=11, colors='white', length=5)
cbar.outline.set_edgecolor('white')

PLOT_NAME = 'stop_mu_calorimetry'
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.pdf', dpi=300)
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.png', dpi=300)